# Lab Test Trends

Builds the gold-layer `michelin.gold.lab_test_trends` table from silver sources using the workbook mapping. This notebook performs a full rebuild each run and computes month-over-month abnormal-rate trends by `test_name`.

In [0]:
spark.sql("SELECT current_user()").show(truncate=False)

In [0]:
catalog_name = "michelin"
target_table = f"{catalog_name}.gold.lab_test_trends"
source_tables = [
    (catalog_name, "silver", "lab_results"),
    (catalog_name, "silver", "dim_date"),
]


def table_is_listed(catalog_name: str, schema_name: str, table_name: str) -> bool:
    return (
        spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name} LIKE '{table_name}'").count() > 0
    )


missing_tables = [
    f"{catalog_name}.{schema_name}.{table_name}"
    for catalog_name, schema_name, table_name in source_tables
    if not table_is_listed(catalog_name, schema_name, table_name)
]

if missing_tables:
    print("Skipped build because these required source tables are not available yet:")
    for table_name in missing_tables:
        print(f" - {table_name}")
else:
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")
        spark.sql(
            f"""
            CREATE OR REPLACE TABLE {target_table} AS
            WITH monthly_aggregates AS (
              SELECT
                lr.test_name,
                YEAR(CAST(lr.test_date AS DATE)) AS year,
                MONTH(CAST(lr.test_date AS DATE)) AS month,
                CAST(DATE_TRUNC('month', CAST(lr.test_date AS DATE)) AS DATE) AS month_start_date,
                CAST(COUNT(*) AS INT) AS total_tests,
                CAST(SUM(CASE WHEN lr.result_status = 'Abnormal' THEN 1 ELSE 0 END) AS INT) AS abnormal_test_count,
                CAST(SUM(CASE WHEN lr.result_status = 'Critical' THEN 1 ELSE 0 END) AS INT) AS critical_test_count
              FROM {catalog_name}.silver.lab_results AS lr
              GROUP BY
                lr.test_name,
                YEAR(CAST(lr.test_date AS DATE)),
                MONTH(CAST(lr.test_date AS DATE)),
                CAST(DATE_TRUNC('month', CAST(lr.test_date AS DATE)) AS DATE)
            ), enriched AS (
              SELECT
                ma.test_name,
                ma.year,
                ma.month,
                dd.month_name,
                dd.quarter_name,
                ma.total_tests,
                ma.abnormal_test_count,
                ma.critical_test_count,
                CAST(ROUND(CASE WHEN ma.total_tests = 0 THEN NULL ELSE ma.abnormal_test_count * 100.0 / ma.total_tests END, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
                CAST(ROUND(CASE WHEN ma.total_tests = 0 THEN NULL ELSE ma.critical_test_count * 100.0 / ma.total_tests END, 2) AS DECIMAL(5,2)) AS critical_rate_pct
              FROM monthly_aggregates AS ma
              LEFT JOIN {catalog_name}.silver.dim_date AS dd
                ON ma.month_start_date = dd.calendar_date
            ), final AS (
              SELECT
                test_name,
                year,
                month,
                month_name,
                quarter_name,
                total_tests,
                abnormal_test_count,
                critical_test_count,
                abnormal_rate_pct,
                critical_rate_pct,
                CAST(LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct
              FROM enriched
            )
            SELECT
              test_name,
              year,
              month,
              month_name,
              quarter_name,
              total_tests,
              abnormal_test_count,
              critical_test_count,
              abnormal_rate_pct,
              critical_rate_pct,
              prev_month_abnormal_rate_pct,
              CAST(
                ROUND(
                  CASE
                    WHEN prev_month_abnormal_rate_pct IS NULL THEN NULL
                    ELSE abnormal_rate_pct - prev_month_abnormal_rate_pct
                  END,
                  2
                ) AS DECIMAL(6,2)
              ) AS abnormal_rate_change_pct
            FROM final
            """
        )
        print(f"Created {target_table}")
    except Exception as exc:
        print(f"Build could not be completed for {target_table}.")
        print(str(exc))

In [0]:
target_table = "michelin.gold.lab_test_trends"

if spark.catalog.tableExists(target_table):
    preview_df = spark.table(target_table).orderBy("year", "month", "test_name", ascending=[False, False, True])
    display(preview_df.limit(50))
    print(f"Total rows: {preview_df.count():,}")
else:
    print(f"Table not created yet: {target_table}")